In [0]:
# Imports and Variable Set Up
import os
import logging
import uuid
import time
import requests
import json
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from databricks.vector_search.client import VectorSearchClient
from openai import OpenAI
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Catalog / Schema / Volume
catalog = "workspace"
schema = "ai_project"
volume = "raw_data"

# Base Volume Path
vol_path = f"/Volumes/{catalog}/{schema}/{volume}/"

# Books paths
books_landing_path = vol_path + "books/raw"
books_processed_path = vol_path + "books/processed"
book_chunks_table = f"{catalog}.{schema}.book_chunks"

# Docs paths
docs_landing_path = vol_path + "docs/raw"
docs_processed_path = vol_path + "docs/processed"
doc_chunks_table = f"{catalog}.{schema}.doc_chunks"

# AI Gateway / Model config
base_url = "https://7474648118426063.ai-gateway.cloud.databricks.com/mlflow/v1"
embedding_model = "databricks-bge-large-en"
llm_model = "databricks-meta-llama-3-1-405b-instruct"

# Client Setup
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=base_url
)

# Vector Search config
vsc = VectorSearchClient()
endpoint_name = "search_endpoint"
book_index_name = f"{catalog}.{schema}.book_vector_index"
doc_index_name = f"{catalog}.{schema}.doc_vector_index"

# File config
valid_extensions = ('.txt', '.pdf')
source_config = {
    "books": {
        "landing_path": books_landing_path,
        "processed_path": books_processed_path,
        "table": book_chunks_table,
        "min_size_kb": 10,
        "index": book_index_name
    },
    "docs": {
        "landing_path": docs_landing_path,
        "processed_path": docs_processed_path,
        "table": doc_chunks_table,
        "min_size_kb": 1,
        "index": doc_index_name
    }
}

# %pip install -r https://raw.githubusercontent.com/seaninc-training/databricks-ai-project/refs/heads/main/requirements.txt
# dbutils.library.restartPython()

In [0]:
%run ./utils/logging_utils

In [0]:
%run ./utils/search_utils

In [0]:
all_files_to_process = {}

def get_files_to_process(landing_path, source_type):

    if source_type is None:
        raise ValueError("source_type must be specified: 'books' or 'docs'")
    elif source_type not in source_config:
        raise ValueError(f"Invalid source_type '{source_type}'. Must be one of: {list(source_config.keys())}")
    
    to_process = []
    min_size_kb = source_config[source_type]["min_size_kb"]
    
    try:
        raw_files = dbutils.fs.ls(landing_path)
        if not raw_files:
            logger.warning(f"⚠️ Source folder is empty: {landing_path}")
        else:
            logger.info(f"✅ Found {len(raw_files)} files to process.")
            for file in raw_files:
                if file.name.lower().endswith(valid_extensions):
                    size_kb = file.size / 1024
                    if size_kb < min_size_kb:
                        logger.error(f"❌ Skipping {file.name}: File is too small ({size_kb:.2f} KB).")
                        continue
                    to_process.append({
                        "path": file.path,
                        "name": file.name,
                        "type": "pdf" if file.name.lower().endswith(".pdf") else "text",
                        "source_type": source_type
                    })
                    logger.info(f"📖 {file.name} validated ({size_kb:.2f} KB).")
                else:
                    logger.warning(f"⚠️  {file} is not a permitted file type.")

            if len(to_process) == 0:
                logger.warning(f"⚠️ No valid files found in {landing_path}.")
            else:
                logger.info(f"✅ Total files ready for ingestion: {len(to_process)}")

    except Exception as e:
        logger.error(f"❌ Error accessing volume: {e}")
    
    return to_process


for source_type, config in source_config.items():
    all_files_to_process[source_type] = get_files_to_process(config["landing_path"], source_type)

In [0]:
# Set up text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""])


def process_files(files_to_process, source_type):
    config = source_config[source_type]
    chunks_table = config["table"]
    processed_path = config["processed_path"]

    for file_info in files_to_process:
        try:
            logger.info(f"🚀 Processing: {file_info['name']}")

            local_path = file_info['path'].replace("dbfs:", "")
            
            if file_info['type'] == "pdf":
                loader = PyPDFLoader(local_path)
            else:
                loader = TextLoader(local_path, encoding="utf-8")

            raw_docs = loader.load()

            # Create Chunks
            chunks = text_splitter.split_documents(raw_docs)

            # Define explicit schema to prevent type inference issues
            chunks_schema = StructType([
                StructField("chunk_id", StringType(), False),
                StructField("content", StringType(), True),
                StructField("source", StringType(), True),
                StructField("type", StringType(), True),
                StructField("page_number", IntegerType(), True),
                StructField("start_index", IntegerType(), True)
            ])
            
            # Prepare data for Vector Search
            data = [{
                "chunk_id": str(uuid.uuid4()),
                "content": chunk.page_content,
                "source": file_info['name'],
                "type": file_info['type'],
                "page_number": int(chunk.metadata.get("page", 1)),
                "start_index": int(chunk.metadata.get("start_index", 0))
            } for chunk in chunks]

            # Convert to Spark DF with explicit schema
            df = spark.createDataFrame(data, schema=chunks_schema)

            # Write to Delta with CDF Enabled
            if not spark.catalog.tableExists(chunks_table):
                (df.write.format("delta")
                   .option("delta.enableChangeDataFeed", "true")
                   .mode("overwrite")
                   .saveAsTable(chunks_table))
                logger.info(f"✨ Created new table: {chunks_table}")
            else:
                df.write.format("delta").mode("append").saveAsTable(chunks_table)
                logger.info(f"➕ Appended {len(chunks)} chunks to {chunks_table}")

            # Move file to processed folder
            destination = f"{processed_path}/{file_info['name']}"
            dbutils.fs.mv(local_path, destination)
            logger.info(f"✅ Processed and moved: {file_info['name']}")

            # Log success
            write_log(
                file_name=file_info['name'],
                source_type=source_type,
                target_table=chunks_table,
                status="SUCCESS",
                chunk_count=len(chunks)
            )

        except Exception as e:
            logger.error(f"❌ Failed to process {file_info['name']}: {e}")
            # Log failure

            write_log(
                file_name=file_info['name'],
                source_type=source_type,
                target_table=chunks_table,
                status="FAILED",
                error_message=str(e)
            )

    logger.info(f"🏁 All files processed for table: {chunks_table}")


for source_type, files in all_files_to_process.items():
    if files:
        process_files(files, source_type)
    else:
        logger.info(f"ℹ️ No files to process for source type: {source_type}")

In [0]:
# Idempotent Vector Search Setup

# 1. Ensure Endpoint exists
if not any(e['name'] == endpoint_name for e in vsc.list_endpoints().get('endpoints', [])):
    logger.info(f"🚀 Creating endpoint '{endpoint_name}'...")
    try:
        vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
        vsc.wait_for_endpoint(endpoint_name)
        logger.info(f"🟢 Endpoint '{endpoint_name}' is now ONLINE.")
    except Exception as e:
        logger.error(f"❌ Failed to create endpoint: {e}")
else:
    logger.info(f"✅ Endpoint '{endpoint_name}' is ready.")

# 2. Ensure Indexes exist and sync if needed
for source_type, config in source_config.items():
    index = config["index"]
    chunks_table = config["table"]
    files_processed = len(all_files_to_process.get(source_type, []))

    if not vsc.index_exists(endpoint_name=endpoint_name, index_name=index):
        logger.info(f"✨ Creating new index '{index}'...")
        try:
            vsc.create_delta_sync_index(
                endpoint_name=endpoint_name,
                source_table_name=chunks_table,
                index_name=index,
                pipeline_type="TRIGGERED",
                primary_key="chunk_id",
                embedding_source_column="content",
                embedding_model_endpoint_name=embedding_model
            )
            logger.info(f"⏳ Waiting for '{index}' to come ONLINE before proceeding...")

            # Wait for this index before creating the next one
            wait_for_index(index)

        except Exception as e:
            logger.error(f"❌ Failed to create index '{index}': {e}")
    else:
        logger.info(f"✅ Index '{index}' already exists.")

        # Smart Sync: Only sync if new files were processed
        if files_processed > 0:
            try:
                logger.info(f"🔄 New data detected ({files_processed} files). Triggering sync for '{index}'...")
                vsc.get_index(endpoint_name, index).sync()

                # Wait for sync to complete before moving to next index
                wait_for_index(index)

            except Exception as e:
                logger.error(f"❌ Failed to sync index '{index}': {e}")
        else:
            logger.info(f"ℹ️ No new files for '{source_type}'. Skipping sync.")

In [0]:
# Diagnostic
# ⚙️ Configure before running

# -> Book Test
# run_diagnostic(
#     source_type="books",
#     query="What was the weapon Raskolnikov used in the crime?"
# )

# -> Doc Test
run_diagnostic(
    source_type="docs",
    query="How do I create a managed table in Databricks?"
)

In [0]:
# Cell 8: Agent Setup

# Initialize Chat History
if 'chat_history' not in globals():
    chat_history = []
    logger.info("🧠 Memory Initialized.")

# Tool Schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_documents",
            "description": "Search the vector index for relevant context from books or technical documentation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "The search query."
                    },
                    "source_type": {
                        "type": "string",
                        "enum": ["books", "docs"],
                        "description": "The source domain to search. Use 'books' for literary content and 'docs' for technical documentation."
                    }
                },
                "required": ["question", "source_type"]
            }
        }
    }
]

In [0]:
# Cell 9: Agent

def run_agent(user_query):
    global chat_history

    # 1. System Prompt
    system_prompt = {
        "role": "system",
        "content": """You are an intelligent Data Engineering Assistant with access to technical documentation and literary sources.

        RULES:
        1. Always use the search_documents tool to retrieve context before answering.
        2. Choose source_type carefully:
           - Use 'docs' for technical questions about Databricks, SQL, or data engineering concepts
           - Use 'books' for questions about literary content
        3. Only use retrieved context to form your answer. If nothing relevant is found, state that clearly.
        4. Always cite the Source Page and File Name from retrieved context.
        5. If a definitive answer isn't in the retrieved data, explain what IS there and clarify the uncertainty.
        6. For actions such as creating tables or running queries, always present the generated code to the user and ask for confirmation before executing."""
    }

    # 2. Build conversation thread
    messages = [system_prompt] + chat_history + [{"role": "user", "content": user_query}]

    # 3. Initial Call (Thinking Phase)
    response = client.chat.completions.create(
        model=llm_model,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    # Cost Tracker (Part 1)
    total_tokens = response.usage.total_tokens
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # 4. Tool Execution (Acting Phase)
    if tool_calls:
        for tool_call in tool_calls:
            query_args = json.loads(tool_call.function.arguments)

            # Route to correct source type
            observations = search_documents(
                query=query_args['question'],
                source_type=query_args['source_type']
            )

            # Format observations as string for LLM context
            if observations:
                context_blocks = [
                    f"Source Page {row[2]} (File: {row[1]}, Index: {row[3]}): {row[0]}"
                    for row in observations
                ]
                context = "\n---\n".join(context_blocks)
            else:
                context = "No relevant results found in the index."

            logger.info(f"📡 Retrieved {len(observations)} excerpts from '{query_args['source_type']}' index.")

            # Feed observations back to the Brain
            messages.append(response_message)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "search_documents",
                "content": context
            })

        # Final Generation AFTER all tool calls complete
        final_response = client.chat.completions.create(
            model=llm_model,
            messages=messages
        )

        # Cost Tracker (Part 2)
        total_tokens += final_response.usage.total_tokens
        answer = final_response.choices[0].message.content
    else:
        answer = response_message.content

    # 5. Update Memory
    chat_history.append({"role": "user", "content": user_query})
    chat_history.append({"role": "assistant", "content": answer})

    # 6. Display Response
    print(f"\n{'='*50}")
    print(f"🤖 AGENT RESPONSE:\n{answer}")
    print(f"{'='*50}")
    print(f"💰 USAGE: {total_tokens} tokens")

    return answer

In [0]:
# Cell 10: Run Agent
# run_agent("What weapon did Raskolnikov use to commit the crime?")
run_agent("How do I create a managed table in Databricks?")
